
#Question-answering chatbot using a pre-trained language model



Install and Import Required Libraries

In [1]:
!pip install transformers torch accelerate sentencepiece -q

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


Using FLAN-T5 Pre-trained Model

In [3]:
print("\n" + "="*50)
print("LOADING FLAN-T5 MODEL...")
print("="*50)


LOADING FLAN-T5 MODEL...


Load model and tokenizer

In [4]:
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"✓ Model '{model_name}' loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✓ Model 'google/flan-t5-base' loaded successfully!


Simple inference function

In [5]:
def answer_question(question, context=""):
    """
    Answer a question using FLAN-T5

    Args:
        question: The question to answer
        context: Optional context for the question
    """
    if context:
        input_text = f"Answer the question based on the context.\nContext: {context}\nQuestion: {question}"
    else:
        input_text = f"Question: {question}"

    # Tokenize input
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

    # Generate answer
    outputs = model.generate(
        inputs.input_ids,
        max_length=150,
        num_beams=4,
        early_stopping=True,
        temperature=0.7
    )

    # Decode answer
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

Testing the Chatbot

In [6]:
print("\n" + "="*50)
print("TESTING CHATBOT")
print("="*50)


TESTING CHATBOT


In [7]:
# Test 1: General knowledge question
question1 = "What is the capital of France?"
answer1 = answer_question(question1)
print(f"\nQ: {question1}")
print(f"A: {answer1}")

# Test 2: Math question
question2 = "What is 25 + 37?"
answer2 = answer_question(question2)
print(f"\nQ: {question2}")
print(f"A: {answer2}")

# Test 3: Context-based question
context = "Machine learning is a subset of artificial intelligence that focuses on building systems that can learn from data. It uses algorithms to identify patterns and make decisions with minimal human intervention."
question3 = "What is machine learning?"
answer3 = answer_question(question3, context)
print(f"\nContext: {context}")
print(f"Q: {question3}")
print(f"A: {answer3}")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Q: What is the capital of France?
A: monctro

Q: What is 25 + 37?
A: 0

Context: Machine learning is a subset of artificial intelligence that focuses on building systems that can learn from data. It uses algorithms to identify patterns and make decisions with minimal human intervention.
Q: What is machine learning?
A: building systems that can learn from data


Interactive chatbot

In [10]:
def run_chatbot():
    """
    Interactive chatbot session
    """
    print("\n" + "="*50)
    print("INTERACTIVE Q&A CHATBOT")
    print("="*50)
    print("Type 'quit' or 'exit' to end the conversation")
    print("Type 'context: <your context>' to set context for questions")
    print("="*50 + "\n")

    current_context = ""

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Chatbot: Goodbye! Have a great day!")
            break

        if user_input.lower().startswith('context:'):
            current_context = user_input[8:].strip()
            print(f"Chatbot: Context set! Now ask me questions about it.\n")
            continue

        if not user_input:
            continue

        # Get answer
        answer = answer_question(user_input, current_context)
        print(f"Chatbot: {answer}\n")

# Run the chatbot
run_chatbot()


INTERACTIVE Q&A CHATBOT
Type 'quit' or 'exit' to end the conversation
Type 'context: <your context>' to set context for questions

You: What is AI?
Chatbot: artificial intelligence (AI)

You: What does AI do?
Chatbot: identifier

You: Context: Artificial Intelligence (AI) is a branch of computer science that enables machines to simulate human intelligence, including learning, reasoning, problem-solving, and decision-making. These systems analyze vast datasets to identify patterns and act independently, often replacing the need for human intervention.
Chatbot: Context set! Now ask me questions about it.

You: What does AI do?
Chatbot: identify patterns and act independently

You: What is machine learning?
Chatbot: Artificial Intelligence

You: What is AI?
Chatbot: branch of computer science

You: Quit
Chatbot: Goodbye! Have a great day!
